# 개인CB(폴더9) 4개년 결합 — 경량(메모리 최적화) EDA·검증 노트북 v2

**v1 대비 변경 (RAM 초과 대응)**
1. **컬럼 대폭 축소**: 157개 중 **29개만 사용 (128개 = 81.5% 제외)**. 선정 기준은 아래 STEP 1에 명시.
2. **전 컬럼 청크 스캔(구 STEP 1) 제거** — RAM·시간의 최대 소비처였음. 결측·센티널 진단은 4만행 샘플 분석에서 이미 완료(별도 보고서)했고, 본 노트북에서는 사용 컬럼 29개에 대해서만 로드 후 확인.
3. **dtype을 로드 시점에 고정**: 수치형 float32, 코드형 int8, ID는 결합 후 category 변환 → 13M행 기준 약 1.5GB 이내로 억제.
4. 각 STEP 종료 시 중간 객체 `del` + `gc.collect()`.

> 그래도 부족하면: 런타임 유형을 '하이램'으로 변경하거나, STEP 2의 `SAMPLE_FRAC`으로 연도별 표본 비율을 낮춰 파일럿 실행.

## STEP 0. Drive 마운트 및 경로 설정

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

DATA_DIR = '/content/drive/MyDrive/09.개인_CB정보'
FILES = {2019: '201912_개인CB.csv',
         2020: '202012_개인CB.csv',
         2021: '202112_개인CB.csv',
         2022: '202212_개인CB.csv'}
PATHS = {y: os.path.join(DATA_DIR, f) for y, f in FILES.items()}
for y, p in PATHS.items():
    ok = os.path.exists(p)
    print(f'{y}: {"OK" if ok else "❌ 없음"}  {os.path.getsize(p)/1e6:,.0f} MB' if ok else f'{y}: ❌ 없음  {p}')

RANDOM_STATE = 42
SAMPLE_FRAC  = 1.0   # RAM 부족 시 0.3 등으로 낮춰 파일럿 실행

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2019: OK  1,491 MB
2020: OK  1,491 MB
2021: OK  1,492 MB
2022: OK  1,493 MB


## STEP 1. 사용 컬럼 확정 (29 / 157개, 81.5% 제외) — 선정 근거 명시

| 그룹 | 컬럼 | 선정 근거 |
|---|---|---|
| 키·인구 (3) | `ID`, `GENDER`, `AGE_BAND` | 결합키·그룹분할키 / 기본 인구 피처 (AGE_BAND는 베이스라인 중요도 상위권) |
| 씬파일러 기본 정의 (5) | `C1M210000`, `C18233003`, `C18233004`, `C18233005`, `L10220000` | 세그먼테이션 정의 그 자체 |
| 씬파일러 엄격 정의 (3) | `C1L120004`, `L10173000`, `D10110000` | 정의 + 베이스라인 중요도 2·3·5위 |
| 카드·대출 활동 (6) | `C1L120161`, `L10210000`, `C1M210003`, `C1M210006`, `L10210003`, `L10210006` | `C1L120161` 중요도 1위 / 나머지는 신규개설 카운트(진입자 시점 피처) 및 중요도 상위 |
| 연체 D계열 (6) | `D1011000C`, `D10110003`, `D10110006`, `D10210D00`, `D10232000`, `D1013A000` | PERF2 전방성 검증에 필수(`D1011000C`) + 중요도 4~12위 |
| 확정 대안변수 (3) | `AL012G005`, `AL012G011`, `AL012G019` | 팀 확정 사용. 샘플 IV 0.46 / 0.11 / 0.48 |
| 타겟·참조 (3) | `PERF2`, `PERF3`, `SCORE` | 주 타겟 / 익스포저 필터 / SCORE 적정성 재검토·참조값 |

**제외한 128개 컬럼과 사유** (대분류):
- 활성카드 한도·소진율류 `C1L1200xx` 대부분 (결측 53~73% + 센티널 8888888.8 오염, 중요도 하위)
- 업종별 대출 세분류 `L90xxx`/`L102xxx` 세부 (총건수·기관수와 중복 정보, 씬파일러에선 전부 0)
- 연체 D계열 세부 시점 변형들 (대표 6개로 충분, 상호 상관 높음)
- 자산평가 U계열 (샘플 중요도 하위 — 추후 확장 후보로만)
- 차량 AL0C 5종 (-9 정보없음 88%), `AS120G001`(-9 56%)
- 상수 컬럼 7종: `C1M210001`, `L90216306`, `L10218800`, `L10218B00`, `U10000002`, `AP0910001`, `AP0910002`
- `STDT`(파일명으로 연도 태깅하므로 불필요), `SCORE_6M`, `PERF1`·`PERF4`(PERF2에 100% 포함됨을 이미 검증), 일자 문자열 `D10147000`·`D10155000`

In [3]:
# ===== 사용 컬럼 정의 (29개) =====
KEY_DEMO   = ['ID', 'GENDER', 'AGE_BAND']
TF_BASE5   = ['C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000']
TF_STRICT3 = ['C1L120004', 'L10173000', 'D10110000']
ACTIVITY   = ['C1L120161', 'L10210000', 'C1M210003', 'C1M210006', 'L10210003', 'L10210006']
DELINQ     = ['D1011000C', 'D10110003', 'D10110006', 'D10210D00', 'D10232000', 'D1013A000']
AL3        = ['AL012G005', 'AL012G011', 'AL012G019']       # 확정 사용
TARGET_REF = ['PERF2', 'PERF3', 'SCORE']

USE_COLS = KEY_DEMO + TF_BASE5 + TF_STRICT3 + ACTIVITY + DELINQ + AL3 + TARGET_REF
assert len(USE_COLS) == 29, len(USE_COLS)
print(f'사용 컬럼 {len(USE_COLS)}개 / 원본 157개 → 제외 {157-len(USE_COLS)}개 ({(157-len(USE_COLS))/157:.1%})')
print(USE_COLS)

# 로드 시점 dtype 고정 (float64 중간 단계 자체를 회피)
DTYPES = {c: 'float32' for c in USE_COLS}
DTYPES['ID'] = 'str'
INT8_COLS = ['GENDER', 'AGE_BAND', 'PERF2', 'PERF3'] + AL3   # 로드 후 int8 변환

사용 컬럼 29개 / 원본 157개 → 제외 128개 (81.5%)
['ID', 'GENDER', 'AGE_BAND', 'C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000', 'C1L120004', 'L10173000', 'D10110000', 'C1L120161', 'L10210000', 'C1M210003', 'C1M210006', 'L10210003', 'L10210006', 'D1011000C', 'D10110003', 'D10110006', 'D10210D00', 'D10232000', 'D1013A000', 'AL012G005', 'AL012G011', 'AL012G019', 'PERF2', 'PERF3', 'SCORE']


## STEP 2. 4개년 파일 결합 로드 (메모리 최적화)
- `usecols=29개` + `dtype=float32`로 읽고, 파일명 기준 `YEAR` 태깅.
- 코드형 컬럼은 int8, `ID`는 결합 후 category로 변환 (문자열 13M개 → 정수 코드).
- 파일 하나 처리할 때마다 즉시 메모리 해제.

In [4]:
def read_year(path, year):
    for enc in ('utf-8', 'utf-8-sig', 'cp949'):
        try:
            d = pd.read_csv(path, encoding=enc,
                            usecols=lambda c: c in USE_COLS, dtype=DTYPES)
            break
        except UnicodeDecodeError:
            continue
    if SAMPLE_FRAC < 1.0:
        d = d.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
    d['YEAR'] = np.int16(year)
    for c in INT8_COLS:
        if c in d and d[c].notna().all():
            d[c] = d[c].astype('int8')
    return d

parts = []
for y, p in PATHS.items():
    d = read_year(p, y)
    print(f'{y}: {len(d):,}행  ({d.memory_usage(deep=True).sum()/1e9:.2f} GB)')
    parts.append(d)

df = pd.concat(parts, ignore_index=True)
del parts, d; gc.collect()

df['ID'] = df['ID'].astype('category')     # 문자열 → 정수코드
print(f'\n결합: {df.shape[0]:,}행 × {df.shape[1]}열, 총 메모리 {df.memory_usage(deep=True).sum()/1e9:.2f} GB')

# 사용 컬럼 29개에 대해서만 결측 확인 (전 컬럼 스캔 대체)
na = df.isna().mean().sort_values(ascending=False)
print('\n[사용 컬럼 결측 비율 > 0]')
print((na[na > 0] * 100).round(2).to_string() if (na > 0).any() else ' 결측 없음')

2019: 3,129,036행  (0.64 GB)
2020: 3,129,036행  (0.64 GB)
2021: 3,129,036행  (0.64 GB)
2022: 3,129,036행  (0.64 GB)

결합: 12,516,144행 × 30열, 총 메모리 1.64 GB

[사용 컬럼 결측 비율 > 0]
 결측 없음


## STEP 3. 패널 구조 검증 — ID 등장 횟수 분포
상위 1만 추출본에서 보였던 "전 ID 4회 등장"이 원본에서도 성립하는지 확인한다. 결과에 따라 전이분석·Δ피처·전방성 검증의 모집단(2개년 이상 등장 ID)이 정해진다.

In [5]:
appear = df.groupby('ID', observed=True)['YEAR'].nunique()
print(f'고유 ID 수: {len(appear):,}')
dist = appear.value_counts().sort_index()
display(dist.rename('ID 수').to_frame().assign(비율=(dist/dist.sum()).round(4)))

multi = appear[appear >= 2]
print(f'2개 연도 이상 등장 ID: {len(multi):,} ({len(multi)/len(appear):.1%}) ← 패널 분석 모집단')
del appear, dist; gc.collect()

고유 ID 수: 3,129,036


,ID 수,비율
YEAR,,
4,3129036,1.0


2개 연도 이상 등장 ID: 3,129,036 (100.0%) ← 패널 분석 모집단


38

## STEP 4. 씬파일러 정의(기본/엄격), 클래스 불균형, 진입자 코호트
- 기본: 5개 필드 모두 0 / 엄격: +3개 필드 0 또는 Null
- 전체·기본TF·엄격TF × 연도별 PERF2 분포 → EPV 기준 지도학습 가능성 판정
- 기본TF 양성의 과거이력 보유 비율(폐계좌 오분류 검증)
- 전년 기본TF → 당해 이탈 = **신규 신용진입자** 코호트 산출 (주 타겟 모집단)

In [6]:
df['TF_BASE']   = (df[TF_BASE5].fillna(0) == 0).all(axis=1)
df['TF_STRICT'] = df['TF_BASE'] & (df[TF_STRICT3].fillna(0) == 0).all(axis=1)

def report(mask, name):
    sub = df.loc[mask]
    n, pos = len(sub), int(sub['PERF2'].sum())
    print(f'{name:>8s} | n={n:>12,} | PERF2=1: {pos:>8,} ({pos/max(n,1)*100:.4f}%)')
    return sub.groupby('YEAR')['PERF2'].agg(n='size', pos='sum', rate=lambda s: s.mean()*100).round(4)

print('=== 집단별 PERF2 불균형 ===')
tabs = {'전체': report(df.index == df.index, '전체'),
        '기본TF': report(df['TF_BASE'], '기본TF'),
        '엄격TF': report(df['TF_STRICT'], '엄격TF')}
display(pd.concat(tabs, axis=1))

pos_tf = df[df['TF_BASE'] & (df['PERF2'] == 1)]
if len(pos_tf):
    print(f'[기본TF & PERF2=1 ({len(pos_tf):,}건)의 과거이력 보유 비율]')
    display(pd.Series({
        '과거대출(L10173000>0)': (pos_tf['L10173000'].fillna(0) > 0).mean(),
        '과거카드(C1L120004>0)': (pos_tf['C1L120004'].fillna(0) > 0).mean(),
        '연체이력(D10110000>0)': (pos_tf['D10110000'].fillna(0) > 0).mean(),
    }).round(3).rename('비율').to_frame())
    print('→ 높을수록 "기본 정의 양성 = 폐계좌 보유자" → 엄격 정의 채택 근거')

# --- 진입자 코호트 ---
pan = df[['ID','YEAR','TF_BASE','TF_STRICT']].sort_values(['ID','YEAR'])
prev_tf   = pan.groupby('ID', observed=True)['TF_BASE'].shift(1)
prev_year = pan.groupby('ID', observed=True)['YEAR'].shift(1)
valid = prev_year == pan['YEAR'] - 1
df['IS_ENTRANT'] = False
df.loc[pan.index[valid & (prev_tf == True) & (~pan['TF_BASE'])], 'IS_ENTRANT'] = True

ent = df[df['IS_ENTRANT']]
print(f'\n신규진입자: {len(ent):,}행 | PERF2=1: {int(ent.PERF2.sum()):,} ({ent.PERF2.mean()*100:.3f}%)')
display(ent.groupby('YEAR')['PERF2'].agg(n='size', pos='sum', rate=lambda s: s.mean()*100).round(3))

for col in ['TF_BASE','TF_STRICT']:
    was = pan.groupby('ID', observed=True)[col].shift(1)
    keep = df.loc[pan.index[valid & (was == True)], col].mean()
    print(f'{col} 연간 유지율: {keep:.3f}')
del pan, prev_tf, prev_year, valid, pos_tf; gc.collect()

=== 집단별 PERF2 불균형 ===
      전체 | n=  12,516,144 | PERF2=1:  125,472 (1.0025%)
    기본TF | n=   3,254,230 | PERF2=1:    4,668 (0.1434%)
    엄격TF | n=   1,300,233 | PERF2=1:      269 (0.0207%)


전체                   기본TF                  엄격TF             
            n    pos    rate       n   pos    rate       n  pos    rate
YEAR                                                                   
2019  3129036  63813  2.0394  802882  2485  0.3095  290647  248  0.0853
2020  3129036  30773  0.9835  823170  1179  0.1432  396766   12  0.0030
2021  3129036  17933  0.5731  833405   590  0.0708  279249    8  0.0029
2022  3129036  12953  0.4140  794773   414  0.0521  333571    1  0.0003

[기본TF & PERF2=1 (4,668건)의 과거이력 보유 비율]


,비율
과거대출(L10173000>0),0.558
과거카드(C1L120004>0),0.434
연체이력(D10110000>0),0.765


→ 높을수록 "기본 정의 양성 = 폐계좌 보유자" → 엄격 정의 채택 근거

신규진입자: 68,677행 | PERF2=1: 96 (0.140%)


,n,pos,rate
YEAR,,,
2020,8150,43,0.528
2021,8610,7,0.081
2022,51917,46,0.089


TF_BASE 연간 유지율: 0.972
TF_STRICT 연간 유지율: 0.629


93

## STEP 5. PERF2 전방성(forward-looking) 검증
t년 PERF2=1 고객이 t+1년 스냅샷의 `D1011000C`(1년내 발생 연체건수)>0일 비율을 PERF2=0과 비교. 리프트가 크고 같은해 일치율이 낮으면 "미래 12개월 성과창"으로 확정.

In [ ]:
p2 = df[['ID','YEAR','PERF2','D1011000C']].sort_values(['ID','YEAR'])
nxt_d = p2.groupby('ID', observed=True)['D1011000C'].shift(-1)
nxt_y = p2.groupby('ID', observed=True)['YEAR'].shift(-1)
m = (nxt_y == p2['YEAR'] + 1) & nxt_d.notna()

fwd  = pd.Series(nxt_d[m].gt(0).values, index=p2.index[m]).groupby(p2.loc[m,'PERF2']).mean()
same = df.groupby('PERF2')['D1011000C'].apply(lambda s: (s.fillna(0) > 0).mean())
display(pd.DataFrame({'다음해 D>0': fwd, '같은해 D>0': same}).round(4))
print(f'전방성 리프트: {fwd.get(1,np.nan)/max(fwd.get(0,np.nan),1e-9):.1f}배 '
      '(수 배~수십 배면 전방 성과창 확정)')
del p2, nxt_d, nxt_y, m; gc.collect()

,다음해 D>0,같은해 D>0
PERF2,,
0,0.0069,0.0079
1,0.1404,0.1279


전방성 리프트: 20.4배 (수 배~수십 배면 전방 성과창 확정)


41

## STEP 6. SCORE 타겟 적정성 재검토
① SCORE=0(적용배제) 규모와 정체 ② 씬파일러 내 점수 뭉침(IQR) ③ 구간별 PERF2 정합성 → 종합 판정.
순환논리(CB 피처로 KCB 점수 복원 = KCB 모사)와 시점 문제(스냅샷 복원 ≠ 미래 예측)는 데이터와 무관하게 성립하는 개념적 결격 사유.

In [ ]:
zero = df['SCORE'] == 0
print(f'① SCORE=0: {int(zero.sum()):,}행 ({zero.mean()*100:.3f}%)')
if zero.any():
    print(f'   SCORE=0의 PERF2=1 비율: {df.loc[zero,"PERF2"].mean()*100:.2f}% (전체 {df["PERF2"].mean()*100:.2f}%)')
    print(f'   SCORE=0의 연체이력>0 비율: {(df.loc[zero,"D10110000"].fillna(0)>0).mean()*100:.1f}%')
    print('   → 0점=연체자 집단이면 회귀 타겟 사용 시 제외/별도 클래스 필수')

nz = df[~zero]
q = lambda s: s.quantile(.75) - s.quantile(.25)
print(f'\n② SCORE IQR — 전체 {q(nz["SCORE"]):.0f} vs 엄격TF {q(nz.loc[nz.TF_STRICT,"SCORE"]):.0f}'
      '  (TF가 크게 좁으면 점수 뭉침 → 복원 가치 낮음)')

print('\n③ SCORE 구간별 PERF2 (0점 제외)')
bands = pd.cut(nz['SCORE'], [0,400,600,700,800,900,1000])
display(nz.groupby(bands, observed=True)['PERF2'].agg(n='size', pos='sum', rate=lambda s: s.mean()*100).round(3))

print('\n=== 판정: SCORE는 주 타겟 비권장 ===')
print(' 사유: (1) 순환논리 — KCB 모사는 대안평가가 아님  (2) 스냅샷 복원으로 과제 변질')
print('       (3) 위 ①·② 데이터 근거')
print(' 허용 용도: (a) 레퍼런스 복원 "실험" 벤치마크  (b) S0 클러스터 외부 참조값')
del nz, zero, bands; gc.collect()

① SCORE=0: 1,565행 (0.013%)
   SCORE=0의 PERF2=1 비율: 0.96% (전체 1.00%)
   SCORE=0의 연체이력>0 비율: 53.0%
   → 0점=연체자 집단이면 회귀 타겟 사용 시 제외/별도 클래스 필수

② SCORE IQR — 전체 224 vs 엄격TF 72  (TF가 크게 좁으면 점수 뭉침 → 복원 가치 낮음)

③ SCORE 구간별 PERF2 (0점 제외)


,n,pos,rate
SCORE,,,
"(0, 400]",631899,88573,14.017
"(400, 600]",254608,20003,7.856
"(600, 700]",1874711,8789,0.469
"(700, 800]",2124426,4714,0.222
"(800, 900]",2071495,2173,0.105
"(900, 1000]",5557440,1205,0.022



=== 판정: SCORE는 주 타겟 비권장 ===
 사유: (1) 순환논리 — KCB 모사는 대안평가가 아님  (2) 스냅샷 복원으로 과제 변질
       (3) 위 ①·② 데이터 근거
 허용 용도: (a) 레퍼런스 복원 "실험" 벤치마크  (b) S0 클러스터 외부 참조값


31

## STEP 7. 확정 대안변수 AL012G005/011/019 (주소·직장·휴대폰 이력)
1. 커버리지: -9(정보없음) 비율 — 특히 엄격TF에서 (씬파일러에게도 존재해야 대안변수 적격)
2. 세그먼트별 분포
3. 범주별 PERF2 불량률 + IV (0.02 미만 무용 / 0.02~0.1 약 / 0.1~0.3 중 / 0.3+ 강)
4. 진입자 코호트 내 IV (양성 20건 이상일 때)

In [ ]:
AL_LABEL = {'AL012G005':'자택주소 이력','AL012G011':'직장명 이력','AL012G019':'휴대폰번호 이력'}
for c in AL3:
    df[c] = pd.to_numeric(df[c], errors='coerce')

print('1) -9(정보없음) 비율 (%)')
display((pd.DataFrame({
    '전체':   {c: (df[c]==-9).mean() for c in AL3},
    '기본TF': {c: (df.loc[df.TF_BASE,c]==-9).mean() for c in AL3},
    '엄격TF': {c: (df.loc[df.TF_STRICT,c]==-9).mean() for c in AL3},
}).rename(index=AL_LABEL)*100).round(2))

print('\n2) 분포 (%, 전체 vs 엄격TF)')
for c in AL3:
    t = pd.DataFrame({'전체': df[c].value_counts(normalize=True),
                      '엄격TF': df.loc[df.TF_STRICT,c].value_counts(normalize=True)}).sort_index()
    print(f'--- {c} ({AL_LABEL[c]}) ---'); display((t*100).round(2))

def iv_table(frame, col, target='PERF2'):
    g = frame.groupby(col)[target].agg(n='size', bad='sum')
    g['good'] = g['n'] - g['bad']
    tb, tg = g['bad'].sum(), g['good'].sum()
    g['bad_r'], g['good_r'] = (g['bad']+.5)/(tb+.5), (g['good']+.5)/(tg+.5)
    g['WoE'] = np.log(g['good_r']/g['bad_r'])
    g['bad_rate_%'] = g['bad']/g['n']*100
    return g, ((g['good_r']-g['bad_r'])*g['WoE']).sum()

print('\n3) 범주별 불량률 + IV (전체 모집단)')
ivs = {}
for c in AL3:
    tab, iv = iv_table(df, c); ivs[c] = iv
    print(f'--- {c} ({AL_LABEL[c]}) | IV={iv:.4f} ---')
    display(tab[['n','bad','bad_rate_%','WoE']].round(4))

ent = df[df['IS_ENTRANT']]
if int(ent['PERF2'].sum()) >= 20:
    print('\n4) 진입자 코호트 내 IV')
    for c in AL3:
        _, iv = iv_table(ent, c); ivs[f'{c}(entrant)'] = iv
else:
    print(f'\n4) 진입자 양성 {int(ent.PERF2.sum())}건 — IV 계산 부족, 전체 기준으로 대체')
display(pd.Series(ivs).rename('IV').to_frame().round(4))

1) -9(정보없음) 비율 (%)


,전체,기본TF,엄격TF
자택주소 이력,0.0,0.0,0.0
직장명 이력,0.0,0.0,0.0
휴대폰번호 이력,0.0,0.0,0.0



2) 분포 (%, 전체 vs 엄격TF)
--- AL012G005 (자택주소 이력) ---


,전체,엄격TF
AL012G005,,
-9,0.00,0.00
1,27.76,45.58
2,50.70,46.28
3,13.91,6.39
4,4.26,1.30
5,2.31,0.31
6,0.73,0.10
7,0.32,0.04


--- AL012G011 (직장명 이력) ---


...

,n,bad,bad_rate_%,WoE
AL012G011,,,,
-9,164,6,3.6585,-1.3987
1,8647482,71265,0.8241,0.1977
2,3070518,37755,1.2296,-0.2065
3,650584,13376,2.0560,-0.7290
4,120726,2509,2.0783,-0.7402
5,26670,561,2.1035,-0.7532


--- AL012G019 (휴대폰번호 이력) | IV=0.4712 ---


,n,bad,bad_rate_%,WoE
AL012G019,,,,
-9,158,6,3.7975,-1.4373
1,4203707,16203,0.3854,0.9620
2,7394973,75058,1.0150,-0.0125
3,769180,28652,3.7250,-1.3405
4,109718,4112,3.7478,-1.3469
5,38408,1441,3.7518,-1.3483



4) 진입자 코호트 내 IV


,IV
AL012G005,0.4292
AL012G011,0.0860
AL012G019,0.4712
AL012G005(entrant),0.3643
AL012G011(entrant),0.1259
AL012G019(entrant),0.5405


## STEP 8. 누수 방지 분할: StratifiedGroupKFold(ID) + OOT(2022)
같은 ID의 모든 연도 행이 같은 fold에 들어가도록 강제(튜닝용), 최종 평가는 2019~21 학습 / 2022 테스트.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

train_oot = df[df['YEAR'] < 2022]
test_oot  = df[df['YEAR'] == 2022]
print(f'OOT: train {len(train_oot):,}행(양성 {int(train_oot.PERF2.sum()):,}) / '
      f'test {len(test_oot):,}행(양성 {int(test_oot.PERF2.sum()):,})')

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv = list(sgkf.split(np.arange(len(train_oot)), train_oot['PERF2'].values,
                     groups=train_oot['ID'].cat.codes.values))
for i, (tr_i, va_i) in enumerate(cv):
    g = len(set(train_oot.iloc[tr_i]['ID']) & set(train_oot.iloc[va_i]['ID']))
    print(f'fold{i}: train {len(tr_i):,} / valid {len(va_i):,} / '
          f'양성 {int(train_oot.iloc[va_i].PERF2.sum()):,} / ID겹침 {g} (0 정상)')

OOT: train 9,387,108행(양성 112,519) / test 3,129,036행(양성 12,953)
fold0: train 7,509,600 / valid 1,877,508 / 양성 23,052 / ID겹침 0 (0 정상)
fold1: train 7,509,837 / valid 1,877,271 / 양성 22,565 / ID겹침 0 (0 정상)
fold2: train 7,509,753 / valid 1,877,355 / 양성 22,341 / ID겹침 0 (0 정상)
fold3: train 7,509,642 / valid 1,877,466 / 양성 22,079 / ID겹침 0 (0 정상)
fold4: train 7,509,600 / valid 1,877,508 / 양성 22,482 / ID겹침 0 (0 정상)


## STEP 9. 베이스라인 LightGBM
- 모집단: 진입자 코호트(양성 ≥100) → 부족 시 전체 폴백
- 피처: 인구 + **AL012G 3종(확정)** + 활동·연체 컬럼. `SCORE`는 순환논리 방지를 위해 제외
- 불균형: SMOTE 대신 `scale_pos_weight` / 지표: AUPRC(주)+AUROC+top-5% 리프트+부트스트랩 CI

In [ ]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

pop = df[df['IS_ENTRANT']] if int(df.loc[df['IS_ENTRANT'],'PERF2'].sum()) >= 100 else df
pop_name = '진입자 코호트' if pop is not df and len(pop) < len(df) else '전체(폴백)'
print(f'모집단: {pop_name} — {len(pop):,}행, 양성 {int(pop.PERF2.sum()):,}')

NUM_FEATS = TF_BASE5 + TF_STRICT3 + ACTIVITY + DELINQ
X = pop[NUM_FEATS].copy()
for c in ['GENDER','AGE_BAND'] + AL3:
    X = pd.concat([X, pd.get_dummies(pop[c], prefix=c, dummy_na=True)], axis=1)
y = pop['PERF2'].astype('int8').values
print(f'피처 수: {X.shape[1]}')

tr_m = (pop['YEAR'] < 2022).values; te_m = ~tr_m
X_tr, y_tr, X_te, y_te = X[tr_m], y[tr_m], X[te_m], y[te_m]
spw = (y_tr==0).sum() / max((y_tr==1).sum(), 1)
print(f'train {len(y_tr):,} / test {len(y_te):,} / scale_pos_weight={spw:.0f}')

model = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                           scale_pos_weight=spw, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
model.fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]

auprc = average_precision_score(y_te, proba); auroc = roc_auc_score(y_te, proba)
k = max(int(len(y_te)*0.05), 1)
lift5 = y_te[np.argsort(-proba)[:k]].mean() / max(y_te.mean(), 1e-9)
print(f'\n[OOT 2022] AUPRC={auprc:.4f} (기준선={y_te.mean():.4f}) | AUROC={auroc:.4f} | top-5% 리프트={lift5:.1f}배')

rng = np.random.default_rng(RANDOM_STATE); boots=[]
for _ in range(500):
    idx = rng.integers(0, len(y_te), len(y_te))
    if y_te[idx].sum(): boots.append(average_precision_score(y_te[idx], proba[idx]))
print(f'AUPRC 95% CI: [{np.percentile(boots,2.5):.4f}, {np.percentile(boots,97.5):.4f}]')

imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
display(imp.head(15).to_frame('importance'))
al_share = imp[[c for c in imp.index if c.startswith('AL012G')]].sum()/imp.sum()
print(f'AL012G 3종 중요도 비중: {al_share*100:.1f}%')

모집단: 전체(폴백) — 12,516,144행, 양성 125,472
피처 수: 56
train 9,387,108 / test 3,129,036 / scale_pos_weight=82

[OOT 2022] AUPRC=0.8640 (기준선=0.0041) | AUROC=0.9949 | top-5% 리프트=19.5배
AUPRC 95% CI: [0.8590, 0.8693]


,importance
L10173000,1636
C1L120004,1495
C1L120161,1432
D1013A000,757
L10210000,625
C18233005,608
D10110000,523
L10220000,491
C1M210000,362
L10210006,291


AL012G 3종 중요도 비중: 11.6%


## 정리
- 사용 컬럼 29개(제외 128개, 81.5%)와 선정·제외 사유는 STEP 1에 명시.
- RAM이 여전히 부족하면: ① `SAMPLE_FRAC=0.3`으로 파일럿 → 로직 확인 후 하이램 런타임에서 1.0 재실행, ② STEP 9 원핫 대신 LightGBM `categorical_feature` 직접 지정으로 피처 수 축소.
- 다음 단계: 진입자 코호트 확정 → GroupKFold 튜닝 → gap-OOT 병행 보고 → 폴더11 결합 트랙과 비교.

# 폴더9 추가 작업 셀 (STEP 10~14) — 배치·체크포인트·확장 스캔·모델 A

기존 v2 노트북(STEP 0~9) **뒤에 이어붙이는** 셀이다. 단, STEP 10~11만 한 번 실행해 두면 **이후 세션에서는 원본 13M행을 다시 읽을 필요가 없다** — 세션이 끊겨도 STEP 10의 로드 셀부터 재시작하면 된다.

**제안 방식 대비 보완 3가지**
1. 체크포인트는 CSV가 아니라 **parquet**으로 저장한다. CSV는 dtype이 소실되어(float32→float64, category→object) 재로드 시 메모리가 2배 이상 다시 부풀고, 13M행 기준 쓰기/읽기도 수 배 느리다. parquet은 dtype 보존 + 압축 + 컬럼 단위 부분 로드까지 된다.
2. 배치 스캔은 "20개씩 나누기"만으로는 부족하고, **처리 이력(매니페스트)을 Drive에 기록**해서 어느 배치까지 끝났는지 세션이 죽어도 알 수 있게 한다. 완료된 배치는 자동으로 건너뛴다(멱등 실행).
3. 결측 0 채움은 **순서가 중요**하다: `_was_missing` 플래그를 먼저 만들고 → 0을 채운다. 그리고 0을 채우면 안 되는 예외(소진율 센티널 8888888.8/9999999.9 → NaN, SCORE=0은 별도 의미)를 규칙으로 못박는다.

## STEP 10. 체크포인트 인프라 (Drive에 parquet + 매니페스트)
- `CKPT_DIR`: Drive 내 체크포인트 폴더. 세션이 끊겨도 남는다.
- `save_ckpt`/`load_ckpt`: parquet 저장·로드 (dtype 보존).
- `manifest`: 처리 이력 JSON. 배치 스캔 재개(resume)에 사용.
- **재시작 시나리오**: 런타임이 끊기면 → STEP 0(마운트) → 이 셀 → STEP 11의 '로드' 셀만 실행하면 작업 재개.

In [7]:
import os, gc, json, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

CKPT_DIR = '/content/drive/MyDrive/09.개인_CB정보/ckpt'
os.makedirs(CKPT_DIR, exist_ok=True)
MANIFEST = os.path.join(CKPT_DIR, 'manifest.json')

def save_ckpt(frame, name):
    path = os.path.join(CKPT_DIR, f'{name}.parquet')
    frame.to_parquet(path, index=False)
    print(f'[ckpt 저장] {name}: {len(frame):,}행 → {path} ({os.path.getsize(path)/1e6:.1f} MB)')

def load_ckpt(name):
    path = os.path.join(CKPT_DIR, f'{name}.parquet')
    frame = pd.read_parquet(path)
    print(f'[ckpt 로드] {name}: {len(frame):,}행, {frame.memory_usage(deep=True).sum()/1e9:.2f} GB')
    return frame

def ckpt_exists(name):
    return os.path.exists(os.path.join(CKPT_DIR, f'{name}.parquet'))

def manifest_get():
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f: return json.load(f)
    return {'done_batches': [], 'scanned_cols': []}

def manifest_update(**kw):
    m = manifest_get(); m.update(kw)
    with open(MANIFEST, 'w') as f: json.dump(m, f, ensure_ascii=False, indent=1)
    return m

print('체크포인트 디렉토리:', CKPT_DIR)
print('현재 매니페스트:', manifest_get())

체크포인트 디렉토리: /content/drive/MyDrive/09.개인_CB정보/ckpt
현재 매니페스트: {'done_batches': [], 'scanned_cols': []}


## STEP 11. 분석 코호트 추출 → Drive 저장 (원본 재로드 탈출 지점)
STEP 4까지 실행된 `df`(13M행)에서 아래 3개 코호트만 떼어 parquet으로 저장한다. **이후 모든 작업은 이 파일들로만 진행** — 원본 13M행을 다시 읽을 일이 없어진다.
1. `cohort_entrant_ids_allyears`: 진입자 ID의 **전 연도 행**(약 27만 행) — 모델 A-1(t 시점)·A-2(t-1 시점) 겸용
2. `cohort_strict269`: 엄격TF & PERF2=1 (269행) — 케이스 스터디
3. `cohort_tf_strict_sample`: 엄격TF 음성 표본 50만 행 — 모델 B·프로파일 대조군

> 이 셀은 `df`가 메모리에 있는 세션에서 **한 번만** 실행하면 된다. 이미 저장돼 있으면 자동으로 건너뛴다.

In [8]:
if not ckpt_exists('cohort_entrant_ids_allyears'):
    assert 'IS_ENTRANT' in df.columns, 'STEP 4까지 실행된 df가 필요합니다'
    ent_ids = df.loc[df['IS_ENTRANT'], 'ID'].unique()
    ent_all = df[df['ID'].isin(ent_ids)].copy()
    ent_all['ID'] = ent_all['ID'].astype(str)          # parquet 호환 위해 category 해제
    save_ckpt(ent_all, 'cohort_entrant_ids_allyears')

    s269 = df[df['TF_STRICT'] & (df['PERF2'] == 1)].copy()
    s269['ID'] = s269['ID'].astype(str)
    save_ckpt(s269, 'cohort_strict269')

    neg = df[df['TF_STRICT'] & (df['PERF2'] == 0)].sample(n=500_000, random_state=42).copy()
    neg['ID'] = neg['ID'].astype(str)
    save_ckpt(neg, 'cohort_tf_strict_sample')

    # 배치 스캔용 ID 집합도 따로 저장 (원본 필터링에 사용)
    pd.DataFrame({'ID': ent_ids.astype(str)}).to_parquet(os.path.join(CKPT_DIR, 'entrant_id_list.parquet'))
    del ent_all, s269, neg; gc.collect()
else:
    print('이미 저장됨 — 건너뜀')

# ── 재시작 시에는 여기부터 ──
ent_all = load_ckpt('cohort_entrant_ids_allyears')
print('진입자 행(IS_ENTRANT=True):', int(ent_all['IS_ENTRANT'].sum()), '| 양성:', int(ent_all.loc[ent_all.IS_ENTRANT,'PERF2'].sum()))

[ckpt 저장] cohort_entrant_ids_allyears: 273,468행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/cohort_entrant_ids_allyears.parquet (20.6 MB)
[ckpt 저장] cohort_strict269: 269행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/cohort_strict269.parquet (0.0 MB)
[ckpt 저장] cohort_tf_strict_sample: 500,000행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/cohort_tf_strict_sample.parquet (35.4 MB)
[ckpt 로드] cohort_entrant_ids_allyears: 273,468행, 0.06 GB
진입자 행(IS_ENTRANT=True): 68677 | 양성: 96


## STEP 12. 나머지 128개 컬럼 배치 스캔 (20개씩, 중단·재개 가능)
v2에서 제외했던 128개 컬럼에 놓친 신호가 없는지, **진입자 ID의 행만 골라내며** 20개씩 배치로 스캔한다.
- 원본을 청크(30만 행)로 읽되 진입자 ID(약 6.9만 명)의 행만 남긴다 → 배치당 메모리 수십 MB 수준.
- 각 배치 결과는 즉시 `batch_XX.parquet`으로 Drive에 저장 + 매니페스트 기록. **세션이 끊겨도 이 셀만 다시 실행하면 완료된 배치는 건너뛰고 이어서 진행**한다.
- 결측 처리 규칙(팀 표준): ① `_was_missing` 플래그 먼저 생성 ② 구조적 결측(보유 없음)은 0 채움 ③ 예외 — 소진율 4컬럼의 8888888.8/9999999.9는 0이 아니라 NaN(산출불가), 일자형(D10147000 등)은 0 채움 금지.

In [9]:
# 전체 157개 컬럼명 확보 (헤더만 읽기 — 메모리 0)
all_cols = pd.read_csv(PATHS[2019], nrows=0).columns.tolist()
REMAIN = [c for c in all_cols if c not in USE_COLS and c != 'STDT']
print(f'스캔 대상 잔여 컬럼: {len(REMAIN)}개')

RATIO_SENT = {'C1L120049','C1L120237','C1L120250','C1L120292'}   # 8888888.8/9999999.9 → NaN
DATE_COLS  = {'D10147000','D10155000'}                            # 0 채움 금지(문자 일자)

BATCH = 20
batches = [REMAIN[i:i+BATCH] for i in range(0, len(REMAIN), BATCH)]
ent_ids = set(pd.read_parquet(os.path.join(CKPT_DIR, 'entrant_id_list.parquet'))['ID'])

done = set(manifest_get().get('done_batches', []))
for bi, cols in enumerate(batches):
    tag = f'batch_{bi:02d}'
    if tag in done:
        print(f'{tag}: 완료됨 — 건너뜀'); continue
    print(f'{tag}: {len(cols)}개 컬럼 스캔 중... ({cols[0]} ~ {cols[-1]})')
    parts = []
    for y, p in PATHS.items():
        for chunk in pd.read_csv(p, usecols=['ID'] + cols, chunksize=300_000, low_memory=False):
            sub = chunk[chunk['ID'].astype(str).isin(ent_ids)]
            if len(sub):
                sub = sub.copy(); sub['YEAR'] = np.int16(y); parts.append(sub)
            del chunk
        gc.collect()
    bdf = pd.concat(parts, ignore_index=True); del parts

    # 결측 처리 표준 적용
    for c in cols:
        if c in DATE_COLS or not pd.api.types.is_numeric_dtype(bdf[c]):
            continue                                   # 일자·문자형은 그대로 보존
        if c in RATIO_SENT:
            bdf[c] = bdf[c].replace({8888888.8: np.nan, 9999999.9: np.nan})
        if bdf[c].isna().any():
            bdf[f'{c}_was_missing'] = bdf[c].isna().astype('int8')   # ① 플래그 먼저
            bdf[c] = bdf[c].fillna(0)                                 # ② 그 다음 0 채움
        bdf[c] = bdf[c].astype('float32')

    bdf['ID'] = bdf['ID'].astype(str)
    save_ckpt(bdf, tag)
    done.add(tag); manifest_update(done_batches=sorted(done))
    del bdf; gc.collect()

print('\n배치 스캔 완료:', sorted(done))

스캔 대상 잔여 컬럼: 127개
batch_00: 20개 컬럼 스캔 중... (C1Z001373 ~ C1L120084)
[ckpt 저장] batch_00: 273,468행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/batch_00.parquet (21.2 MB)
batch_01: 20개 컬럼 스캔 중... (C1L120163 ~ L1021001P)
[ckpt 저장] batch_01: 273,468행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/batch_01.parquet (20.1 MB)
batch_02: 20개 컬럼 스캔 중... (L1021003P ~ L10216800)
[ckpt 저장] batch_02: 273,468행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/batch_02.parquet (18.8 MB)
batch_03: 20개 컬럼 스캔 중... (L10216806 ~ U81102010)
[ckpt 저장] batch_03: 273,468행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/batch_03.parquet (19.3 MB)
batch_04: 20개 컬럼 스캔 중... (U81103010 ~ D10133800)
[ckpt 저장] batch_04: 273,468행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/batch_04.parquet (23.3 MB)
batch_05: 20개 컬럼 스캔 중... (D1013380F ~ AL0C00004)
[ckpt 저장] batch_05: 273,468행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/batch_05.parquet (18.5 MB)
batch_06: 7개 컬럼 스캔 중... (AL0C00005 ~ PERF4)
[ckpt 저장] batch_06: 273,468행 → /content/drive/MyDrive/09.개인_CB

### STEP 12-1. 배치 결과 통합 + 개선된 IV 스캔 (0-분리 방식, Track B 공용 로직)
Track B에서 확인된 문제 — 값의 절반 이상이 0인 분포에서 qcut 기반 IV가 0으로 붕괴 — 를 반영해, **0 여부를 먼저 분리한 뒤 잔여를 구간화**하는 방식으로 128개 컬럼의 IV를 진입자 코호트에서 계산한다. 상위 컬럼은 모델 A 피처 확장 후보가 된다.

In [10]:
def iv_zero_split(x, y, n_bins=5):
    """0-분리 IV: 0/비0을 먼저 나누고 비0 구간만 분위수 구간화 (Track B 개선 로직 이식)."""
    x = pd.Series(x).astype(float); y = pd.Series(y).astype(int)
    grp = pd.Series('zero', index=x.index)
    nz = x != 0
    if nz.sum() >= n_bins * 20:
        try:
            grp[nz] = pd.qcut(x[nz], q=n_bins, duplicates='drop').astype(str)
        except ValueError:
            grp[nz] = 'nonzero'
    elif nz.any():
        grp[nz] = 'nonzero'
    g = pd.DataFrame({'g': grp, 'y': y}).groupby('g')['y'].agg(n='size', bad='sum')
    g['good'] = g['n'] - g['bad']
    tb, tg = g['bad'].sum(), g['good'].sum()
    if tb == 0 or tg == 0: return 0.0
    br, gr = (g['bad']+.5)/(tb+.5), (g['good']+.5)/(tg+.5)
    return float(((gr-br) * np.log(gr/br)).sum())

# 배치 통합 (진입 시점 행만: IS_ENTRANT 기준 병합)
ent_key = ent_all.loc[ent_all['IS_ENTRANT'], ['ID','YEAR','PERF2']].copy()
ent_key['ID'] = ent_key['ID'].astype(str)

iv_rows = []
for path in sorted(glob.glob(os.path.join(CKPT_DIR, 'batch_*.parquet'))):
    b = pd.read_parquet(path)
    merged = ent_key.merge(b, on=['ID','YEAR'], how='left')
    feat_cols = [c for c in b.columns if c not in ('ID','YEAR') and pd.api.types.is_numeric_dtype(merged[c])]
    for c in feat_cols:
        iv_rows.append({'col': c, 'IV_entrant': iv_zero_split(merged[c].fillna(0), merged['PERF2'])})
    del b, merged; gc.collect()

iv_scan = pd.DataFrame(iv_rows).sort_values('IV_entrant', ascending=False)
save_ckpt(iv_scan, 'iv_scan_remain128_entrant')
print('\n[진입자 코호트 IV 상위 25 — 모델 A 피처 확장 후보]')
display(iv_scan.head(25))
print('※ 주의: 상위권에 카드한도·대출건수류(노출 변수)가 오면 Track B의 IV 인플레이션과 동일 현상 —')
print('   후보 채택 전에 노출 계열 여부를 반드시 분류할 것 (보고서 v3의 3-1 절차)')

[ckpt 저장] iv_scan_remain128_entrant: 138행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/iv_scan_remain128_entrant.parquet (0.0 MB)

[진입자 코호트 IV 상위 25 — 모델 A 피처 확장 후보]


,col,IV_entrant
136,PERF1,5.656748
137,PERF4,4.959093
123,D10220000,3.882167
121,D10155000,3.494585
135,SCORE_6M,3.295739
120,D10147000,3.226851
119,D10142D0T,3.014532
122,D10175000,2.921707
105,D10133000,2.781068
118,D10142D0N,1.852223


※ 주의: 상위권에 카드한도·대출건수류(노출 변수)가 오면 Track B의 IV 인플레이션과 동일 현상 —
   후보 채택 전에 노출 계열 여부를 반드시 분류할 것 (보고서 v3의 3-1 절차)


## STEP 13. 모델 A — 진입자 코호트 학습 (임계값 수정 + A-1/A-2 두 버전)
- **폴백 임계값 100 → 50 수정**: 진입자 양성 96건이 4건 차이로 전체 모집단에 폴백되던 문제 해결.
- **A-1 (진입 시점 t 피처)**: 신규 고객 첫 심사 상황 재현. 인구 + AL012G + 신규개설 규모.
- **A-2 (진입 전 t-1 피처)**: 씬파일러 시절 정보만으로 미래 연체 예측 — 프로젝트 핵심 질문의 직접 검증. 노출 변수가 구조적으로 전부 0이므로 인구 + AL012G만 남는 순수 대안평가.
- 평가: StratifiedGroupKFold(5) CV — 진입자의 76%가 2022년이라 OOT는 참고치로만 출력.

In [11]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score

MIN_POS = 50   # ← 100에서 수정
ent = ent_all[ent_all['IS_ENTRANT']].copy()
assert int(ent['PERF2'].sum()) >= MIN_POS, '진입자 양성 부족'
print(f'모델 A 모집단: {len(ent):,}행, 양성 {int(ent.PERF2.sum()):,} ({ent.PERF2.mean()*100:.3f}%)')

# A-2용: 각 진입자의 t-1(씬파일러 시절) 행을 병합
prev = ent_all.copy(); prev['YEAR'] = prev['YEAR'] + 1
A2_FEATS = ['GENDER','AGE_BAND','AL012G005','AL012G011','AL012G019']
ent = ent.merge(prev[['ID','YEAR'] + A2_FEATS].rename(columns={c: f'{c}_prev' for c in A2_FEATS}),
                on=['ID','YEAR'], how='left')
print('t-1 병합 성공률:', ent['AL012G019_prev'].notna().mean())

def run_cv(X, y, groups, label):
    aps, aucs = [], []
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    for tr_i, va_i in sgkf.split(X, y, groups):
        spw = (y[tr_i]==0).sum() / max((y[tr_i]==1).sum(), 1)
        m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15,
                               min_child_samples=50, scale_pos_weight=spw,
                               random_state=42, n_jobs=-1, verbose=-1)
        m.fit(X.iloc[tr_i], y[tr_i])
        p = m.predict_proba(X.iloc[va_i])[:,1]
        if y[va_i].sum() > 0:
            aps.append(average_precision_score(y[va_i], p)); aucs.append(roc_auc_score(y[va_i], p))
    print(f'[{label}] CV AUPRC {np.mean(aps):.4f}±{np.std(aps):.4f} '
          f'(기준선 {y.mean():.4f}) | AUROC {np.mean(aucs):.4f}±{np.std(aucs):.4f}')
    return m

y = ent['PERF2'].astype(int).values
groups = ent['ID'].values

# A-1: 진입 시점 피처 (소형: EPV 고려 ~10개)
A1_COLS = ['C1M210003','C1M210006','L10210003','L10210006']   # 신규개설 규모
X1 = ent[A1_COLS].copy()
for c in ['GENDER','AGE_BAND'] + AL3:
    X1 = pd.concat([X1, pd.get_dummies(ent[c], prefix=c)], axis=1)
m1 = run_cv(X1, y, groups, 'A-1 진입시점(t)')

# A-2: 진입 전(t-1) 피처만 — 순수 대안평가
X2 = pd.DataFrame(index=ent.index)
for c in A2_FEATS:
    X2 = pd.concat([X2, pd.get_dummies(ent[f'{c}_prev'], prefix=c)], axis=1)
m2 = run_cv(X2.fillna(0), y, groups, 'A-2 진입전(t-1) 순수대안')

print('\n해석 가이드: A-2가 기준선 대비 유의한 리프트를 보이면')
print('"무이력 시점의 대안정보만으로도 미래 연체 변별 가능"이라는 프로젝트 핵심 주장의 직접 증거')

모델 A 모집단: 68,677행, 양성 96 (0.140%)
t-1 병합 성공률: 1.0
[A-1 진입시점(t)] CV AUPRC 0.0022±0.0008 (기준선 0.0014) | AUROC 0.6416±0.0501
[A-2 진입전(t-1) 순수대안] CV AUPRC 0.0028±0.0011 (기준선 0.0014) | AUROC 0.6212±0.0810

해석 가이드: A-2가 기준선 대비 유의한 리프트를 보이면
"무이력 시점의 대안정보만으로도 미래 연체 변별 가능"이라는 프로젝트 핵심 주장의 직접 증거


## STEP 14. 엄격TF 양성 269건 케이스 스터디
"완전 무이력 → 12개월 내 진입+연체"의 실측 사례를 대조군(엄격TF 음성 50만 표본)과 비교 프로파일링한다. 결과는 parquet으로 저장해 보고서·발표에 인용.

In [12]:
s269 = load_ckpt('cohort_strict269')
neg  = load_ckpt('cohort_tf_strict_sample')

prof_cols = ['GENDER','AGE_BAND'] + AL3
rows = {}
for c in prof_cols:
    rows[c] = pd.DataFrame({
        '269 양성(%)': s269[c].value_counts(normalize=True) * 100,
        '엄격TF 음성(%)': neg[c].value_counts(normalize=True) * 100,
    }).round(2)
    print(f'--- {c} ---'); display(rows[c])

print('연도 분포(양성):', s269['YEAR'].value_counts().sort_index().to_dict())
print('→ 2019 편중(248/269)이므로 일반화 주장 금지 — "존재 증명 + 프로파일" 용도로만 서술')

summary = pd.concat(rows, names=['변수','값'])
summary.to_parquet(os.path.join(CKPT_DIR, 'case269_profile.parquet'))
print('프로파일 저장 완료 → ckpt/case269_profile.parquet')

[ckpt 로드] cohort_strict269: 269행, 0.00 GB
[ckpt 로드] cohort_tf_strict_sample: 500,000행, 0.10 GB
--- GENDER ---


,269 양성(%),엄격TF 음성(%)
GENDER,,
1,70.63,50.83
2,29.37,49.17


--- AGE_BAND ---


,269 양성(%),엄격TF 음성(%)
AGE_BAND,,
1,13.75,3.80
2,55.39,48.03
3,5.95,8.15
4,10.41,8.14
5,11.15,10.50
6,2.60,9.69
7,0.74,7.62
8,NaN,3.59
9,NaN,0.49


--- AL012G005 ---


...

--- AL012G011 ---


,269 양성(%),엄격TF 음성(%)
AL012G011,,
-9,NaN,0.00
1,88.48,95.51
2,9.67,4.18
3,1.49,0.25
4,NaN,0.05
5,0.37,0.01


--- AL012G019 ---


,269 양성(%),엄격TF 음성(%)
AL012G019,,
-9,NaN,0.00
1,15.24,46.43
2,56.13,48.42
3,24.54,4.30
4,2.60,0.62
5,1.49,0.23


연도 분포(양성): {2019: 248, 2020: 12, 2021: 8, 2022: 1}
→ 2019 편중(248/269)이므로 일반화 주장 금지 — "존재 증명 + 프로파일" 용도로만 서술
프로파일 저장 완료 → ckpt/case269_profile.parquet


## 세션 재시작 절차 (요약)
1. STEP 0 (Drive 마운트) → STEP 10 (체크포인트 유틸) 실행
2. `ent_all = load_ckpt('cohort_entrant_ids_allyears')` — 원본 13M행 로드 불필요
3. 배치 스캔이 미완이면 STEP 12 재실행(완료 배치 자동 건너뜀), 완료면 STEP 12-1부터
4. 모델·케이스 스터디는 코호트 parquet만으로 동작

# STEP 15. 폴더9 대안변수 검증·확정 (파생변수 설계리포트 v2 연동)

**전제**: STEP 10(유틸) + 11(코호트 parquet) + 12(배치 parquet)가 완료된 상태. STEP 13·14와는 독립이라 순서 무관.
**재시작 세션이라면**: STEP 0(마운트) → STEP 10 → `ent_all = load_ckpt('cohort_entrant_ids_allyears')` → 바로 이 STEP 15 실행 가능. 원본 CSV는 읽지 않는다 — **모든 검증은 배치 parquet(진입자 행만)으로 수행**.

검증 절차는 설계리포트 v2의 채택 기준을 따른다:
1. 원변수 단일 IV 스크리닝 우선 (H8 검증 겸용)
2. 파생변수는 **자기 구성 원변수의 IV를 상회할 때만** 채택 — 못 이기면 폐기하고 비교 결과 자체를 보고
3. -9 / 9(정보없음)는 대체하지 않고 **독립 범주로 유지** (씬파일러는 '정보없음' 자체가 신호)

| 서브스텝 | 내용 |
|---|---|
| 15-0 | 진입자 와이드 프레임 구성 (29컬럼 + 배치 128컬럼 병합) |
| 15-1 | 지정 후보 검증: U81205010·U81303010·U81304010, AL0C00001~5, AS120G001, AP0910001/2 |
| 15-2 | A-파생변수 생성·검증 — **A-5(NONBANK_RATIO)·A-8(DEBIT_ORIENT) 중심**, A-4(UTIL_DELTA)·A-6(ASSET_COVER) 포함, 원변수 대비 IV 비교 |
| 15-3 | 추가 후보 발굴: 잔여 128컬럼 IV 스캔 결과에 노출/대안 계열 태깅 → 비노출 상위 추출 |
| 15-4 | 다중공선성(|r|≥0.8) 대표 선정 + 순환논리(TF 정의와 상관) 검증 → **최종 후보 목록 저장** |

## STEP 15-0. 진입자 와이드 프레임 구성
`ent_all`(29컬럼)에서 진입 시점 행만 추리고, 배치 parquet 전부를 (ID, YEAR)로 병합한다. 약 6.9만 행 × 300여 컬럼 — 수백 MB 이내.

In [13]:
import glob
assert 'ent_all' in dir(), "먼저 STEP 10 실행 후 ent_all = load_ckpt('cohort_entrant_ids_allyears')"

wide = ent_all[ent_all['IS_ENTRANT']].copy()
wide['ID'] = wide['ID'].astype(str)
print(f'진입 시점 행: {len(wide):,} | 양성 {int(wide.PERF2.sum()):,} ({wide.PERF2.mean()*100:.3f}%)')

for path in sorted(glob.glob(os.path.join(CKPT_DIR, 'batch_*.parquet'))):
    b = pd.read_parquet(path); b['ID'] = b['ID'].astype(str)
    add = [c for c in b.columns if c not in wide.columns or c in ('ID','YEAR')]
    wide = wide.merge(b[add], on=['ID','YEAR'], how='left')
print(f'와이드 프레임: {wide.shape[0]:,}행 × {wide.shape[1]}열, {wide.memory_usage(deep=True).sum()/1e9:.2f} GB')
y = wide['PERF2'].astype(int).values

진입 시점 행: 68,677 | 양성 96 (0.140%)
와이드 프레임: 68,677행 × 175열, 0.06 GB


### 공용 IV 함수 (범주형 / 0-분리 연속형)

In [14]:
def iv_cat(x, y, min_n=30):
    """범주형 IV + 범주별 불량률 테이블. 소표본 범주는 '기타'로 병합."""
    x = pd.Series(x).astype(str).fillna('NULL')
    vc = x.value_counts()
    x = x.where(x.map(vc) >= min_n, '기타')
    g = pd.DataFrame({'x': x, 'y': y}).groupby('x')['y'].agg(n='size', bad='sum')
    g['good'] = g['n'] - g['bad']
    tb, tg = g['bad'].sum(), g['good'].sum()
    if tb == 0 or tg == 0: return 0.0, g.assign(bad_rate_pct=0)
    br, gr = (g['bad']+.5)/(tb+.5), (g['good']+.5)/(tg+.5)
    g['bad_rate_pct'] = (g['bad']/g['n']*100).round(3)
    g['WoE'] = np.log(gr/br).round(3)
    return float(((gr-br)*np.log(gr/br)).sum()), g

def iv_num(x, y, n_bins=5):
    """0-분리 연속형 IV (Track B 개선 로직)."""
    x = pd.Series(x).astype(float).fillna(0)
    grp = pd.Series('zero', index=x.index); nz = x != 0
    if nz.sum() >= n_bins*20:
        try: grp[nz] = pd.qcut(x[nz], q=n_bins, duplicates='drop').astype(str)
        except ValueError: grp[nz] = 'nonzero'
    elif nz.any(): grp[nz] = 'nonzero'
    iv, _ = iv_cat(grp, y, min_n=1)
    return iv

def verdict(iv, na_pct):
    if na_pct >= 95: return '사용불가(정보없음 95%+)'
    if iv >= 0.1:   return '채택 후보(중~강)'
    if iv >= 0.02:  return '조건부 채택(약)'
    return '보류(IV<0.02)'

## STEP 15-1. 지정 후보 검증
- **U81205010 / U81303010 / U81304010**: 자산·거주지 평가 신뢰수준 (A=높음/B/C=낮음) — 신뢰수준 자체가 "평가 가능한 자산 정보의 존재·질"을 신호
- **AL0C00001~5**: 차량 정보 5종 (-9=정보없음)
- **AS120G001**: 거주지 평형 구간 (-9=정보없음)
- **AP0910001/2 (A-1·A-2)**: 국민연금·건강보험 성실납부 — 설계리포트 1순위 후보지만, 이전 EDA에서 전값 9(정보없음) 상수로 확인된 바 있어 원본 재판정

판정 기준: 정보없음 95%+ → 사용불가 / IV≥0.1 채택 / 0.02~0.1 조건부 / <0.02 보류. 단, **-9 범주 자체의 WoE가 유의하면 '정보없음 플래그'로의 축약 채택**을 별도 검토(리포트 Part 0-(3) 원칙).

In [15]:
CAND_CAT = {
 'U81205010': '순자산평가 신뢰수준(A/B/C)',
 'U81303010': '거주지시세 신뢰수준(A/B/C)',
 'U81304010': '거주지전세가 신뢰수준(A/B/C)',
 'AL0C00001': '차량 최초구매경과(구간)', 'AL0C00002': '차량 크기', 'AL0C00003': '차량 수입/국산',
 'AL0C00004': '차량 신차/중고', 'AL0C00005': '차량 가격(구간)',
 'AS120G001': '거주지 평형(구간)',
 'AP0910001': '국민연금 성실납부(A-1)', 'AP0910002': '건강보험 성실납부(A-2)',
}
rows = []
for c, label in CAND_CAT.items():
    if c not in wide.columns:
        rows.append({'col': c, '설명': label, '판정': '컬럼 없음'}); continue
    s = wide[c]
    sv = pd.to_numeric(s, errors='coerce')
    if c.startswith(('AL0C','AS1')):
        na_codes = s.isna() | (sv < 0)            # -9, -99999999 등 음수 코드 전부 정보없음
    elif c.startswith('AP09'):
        na_codes = s.isna() | (sv == 9)           # 9 = 정보없음
    else:
        na_codes = s.isna()
    na_pct = na_codes.mean()*100
    iv, tab = iv_cat(s, y)
    rows.append({'col': c, '설명': label, '정보없음(%)': round(na_pct,1),
                 'IV(진입자)': round(iv,4), '판정': verdict(iv, na_pct)})
    print(f'--- {c} {label} | 정보없음 {na_pct:.1f}% | IV {iv:.4f} ---')
    display(tab.sort_index())

res_151 = pd.DataFrame(rows)
print('\n[15-1 판정 요약]'); display(res_151)
save_ckpt(res_151, 'step15_1_지정후보판정')

--- U81205010 순자산평가 신뢰수준(A/B/C) | 정보없음 0.2% | IV 0.0372 ---


,n,bad,good,bad_rate_pct,WoE
x,,,,,
A,36795,52,36743,0.141,-0.015
B,11057,21,11036,0.190,-0.325
C,20720,23,20697,0.111,0.215
None,105,0,105,0.000,-1.214


--- U81303010 거주지시세 신뢰수준(A/B/C) | 정보없음 0.1% | IV 0.0489 ---


,n,bad,good,bad_rate_pct,WoE
x,,,,,
A,43941,63,43878,0.143,-0.028
B,16791,19,16772,0.113,0.191
C,7854,13,7841,0.166,-0.202
None,91,1,90,1.099,-2.466


--- U81304010 거주지전세가 신뢰수준(A/B/C) | 정보없음 0.1% | IV 0.0540 ---


...

--- AP0910002 건강보험 성실납부(A-2) | 정보없음 100.0% | IV 0.0000 ---


,n,bad,good,bad_rate_pct,WoE
x,,,,,
9.0,68677,96,68581,0.14,0.0



[15-1 판정 요약]


,col,설명,정보없음(%),IV(진입자),판정
0,U81205010,순자산평가 신뢰수준(A/B/C),0.2,0.0372,조건부 채택(약)
1,U81303010,거주지시세 신뢰수준(A/B/C),0.1,0.0489,조건부 채택(약)
2,U81304010,거주지전세가 신뢰수준(A/B/C),0.1,0.0540,조건부 채택(약)
3,AL0C00001,차량 최초구매경과(구간),96.9,0.2499,사용불가(정보없음 95%+)
4,AL0C00002,차량 크기,96.9,0.2856,사용불가(정보없음 95%+)
5,AL0C00003,차량 수입/국산,96.7,0.2038,사용불가(정보없음 95%+)
6,AL0C00004,차량 신차/중고,97.1,0.2210,사용불가(정보없음 95%+)
7,AL0C00005,차량 가격(구간),96.7,0.1923,사용불가(정보없음 95%+)
8,AS120G001,거주지 평형(구간),52.9,0.3727,채택 후보(중~강)
9,AP0910001,국민연금 성실납부(A-1),100.0,0.0000,사용불가(정보없음 95%+)


[ckpt 저장] step15_1_지정후보판정: 11행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step15_1_지정후보판정.parquet (0.0 MB)


## STEP 15-2. A-파생변수 생성·검증 (A-5·A-8 중심)
설계리포트의 정의 그대로 생성하고, **파생 IV vs 구성 원변수 최고 IV**를 비교한다. 파생이 원변수를 못 이기면 리포트 원칙대로 폐기(비교 결과는 보고서 콘텐츠로 보존).

| 파생 | 정의 | 원변수 |
|---|---|---|
| A-5 NONBANK_RATIO | (저축은행+보험+할부+카드+기타 대출건수) ÷ 총대출건수 · '대출없음' 별도 범주 | L10210800, L10210B00, L90210300, L90210200, L10210M00 / L10210000 |
| A-8 DEBIT_ORIENT | 체크카드>0 AND 신용카드==0 | C18210000, C1M210000 |
| A-4 UTIL_DELTA | 소진율 1개월(C1L120237) − 6개월(C1L120292), 산출불가 행 제외 | C1L120237, C1L120292 |
| A-4 CA_FLAG | CA 소진율 > 0 | C1L120049 |
| A-6 ASSET_COVER | 순자산평가금액 ÷ (총약정금액+1) | U81202010, L10231000 |

주의: **진입자는 진입 직후라 대출·카드 변수가 살아있는 세그먼트**이므로 A-5·A-8이 계산 가능하다(순수 S0에서는 분모 0). C1Z001373 등 금액 컬럼의 음수는 합성 노이즈 → 0 클리핑.

In [16]:
der = pd.DataFrame(index=wide.index)
comp = []   # (파생명, 파생IV, 원변수 최고IV, 원변수명)

def best_raw_iv(cols):
    ivs = {c: iv_num(wide[c], y) for c in cols if c in wide.columns}
    top = max(ivs, key=ivs.get)
    return ivs[top], top, ivs

# --- A-5 NONBANK_RATIO ---
nb_cols = ['L10210800','L10210B00','L90210300','L90210200','L10210M00']
num = wide[nb_cols].fillna(0).clip(lower=0).sum(axis=1)
den = wide['L10210000'].fillna(0)
ratio = pd.Series('대출없음', index=wide.index)
m = den > 0
ratio[m] = pd.cut((num[m]/den[m]).clip(0,1), [-0.001,0,0.5,1.0], labels=['0','(0,0.5]','(0.5,1]']).astype(str)
iv_a5, tab_a5 = iv_cat(ratio, y)
raw_iv, raw_top, _ = best_raw_iv(nb_cols + ['L10216806'])
comp.append(('A-5 NONBANK_RATIO', iv_a5, raw_iv, raw_top))
print(f'A-5 NONBANK_RATIO IV={iv_a5:.4f} (원변수 최고 {raw_top}={raw_iv:.4f})'); display(tab_a5)

# --- A-8 DEBIT_ORIENT + 원변수 ---
der['DEBIT_ORIENT'] = ((wide['C18210000'].fillna(0)>0) & (wide['C1M210000'].fillna(0)==0)).astype(int)
iv_a8, tab_a8 = iv_cat(der['DEBIT_ORIENT'], y)
raw_iv8, raw_top8, ivs8 = best_raw_iv(['C18210000','C1Z001373'])
comp.append(('A-8 DEBIT_ORIENT', iv_a8, raw_iv8, raw_top8))
print(f'\nA-8 DEBIT_ORIENT IV={iv_a8:.4f} | 원변수: '
      + ', '.join(f'{k} {v:.4f}' for k,v in ivs8.items())
      + f' | DEBIT_ORIENT=1 비율 {der.DEBIT_ORIENT.mean()*100:.1f}%')
display(tab_a8)

# --- A-4 UTIL_DELTA / CA_FLAG (산출불가 행 제외: was_missing 플래그 활용) ---
u1, u6 = wide['C1L120237'], wide['C1L120292']
bad = pd.Series(False, index=wide.index)
for f in ['C1L120237_was_missing','C1L120292_was_missing']:
    if f in wide.columns: bad |= wide[f].fillna(0).astype(bool)
delta = (u1 - u6).where(~bad)
iv_a4 = iv_num(delta.fillna(0), y)
raw_iv4, raw_top4, _ = best_raw_iv(['C1L120237','C1L120292'])
comp.append(('A-4 UTIL_DELTA', iv_a4, raw_iv4, raw_top4))
der['CA_FLAG'] = (wide['C1L120049'].fillna(0) > 0).astype(int)
iv_ca, _ = iv_cat(der['CA_FLAG'], y)
comp.append(('A-4 CA_FLAG', iv_ca, iv_num(wide['C1L120049'], y), 'C1L120049'))
print(f'\nA-4 UTIL_DELTA IV={iv_a4:.4f} (산출가능 {(~bad).mean()*100:.0f}%) | CA_FLAG IV={iv_ca:.4f}')

# --- A-6 ASSET_COVER ---
cover = wide['U81202010'].fillna(0).clip(lower=0) / (wide['L10231000'].fillna(0).clip(lower=0) + 1)
iv_a6 = iv_num(np.log1p(cover), y)
raw_iv6, raw_top6, _ = best_raw_iv(['U81202010','U81204010','L10231000'])
comp.append(('A-6 ASSET_COVER', iv_a6, raw_iv6, raw_top6))
print(f'A-6 ASSET_COVER IV={iv_a6:.4f} (원변수 최고 {raw_top6}={raw_iv6:.4f})')

res_152 = pd.DataFrame(comp, columns=['파생','파생IV','원변수최고IV','원변수'])
res_152['판정'] = np.where(res_152['파생IV'] > res_152['원변수최고IV'],
                           '파생 채택', '원변수 단일 채택(파생 폐기)')
print('\n[15-2 파생 vs 원변수 비교 — 리포트 채택 절차 2단계]'); display(res_152.round(4))
save_ckpt(res_152, 'step15_2_파생검증')

A-5 NONBANK_RATIO IV=0.8858 (원변수 최고 L10210800=0.8301)


,n,bad,good,bad_rate_pct,WoE
x,,,,,
"(0,0.5]",480,14,466,2.917,-3.095
"(0.5,1]",1522,16,1506,1.051,-2.052
0,15564,22,15542,0.141,-0.028
대출없음,51111,44,51067,0.086,0.479



A-8 DEBIT_ORIENT IV=0.1256 | 원변수: C18210000 0.1026, C1Z001373 0.2138 | DEBIT_ORIENT=1 비율 29.8%


,n,bad,good,bad_rate_pct,WoE
x,,,,,
0,48188,51,48137,0.106,0.274
1,20489,45,20444,0.220,-0.458



A-4 UTIL_DELTA IV=0.0773 (산출가능 15%) | CA_FLAG IV=0.0286
A-6 ASSET_COVER IV=0.1002 (원변수 최고 U81202010=0.2260)

[15-2 파생 vs 원변수 비교 — 리포트 채택 절차 2단계]


,파생,파생IV,원변수최고IV,원변수,판정
0,A-5 NONBANK_RATIO,0.8858,0.8301,L10210800,파생 채택
1,A-8 DEBIT_ORIENT,0.1256,0.2138,C1Z001373,원변수 단일 채택(파생 폐기)
2,A-4 UTIL_DELTA,0.0773,0.2568,C1L120292,원변수 단일 채택(파생 폐기)
3,A-4 CA_FLAG,0.0286,0.0615,C1L120049,원변수 단일 채택(파생 폐기)
4,A-6 ASSET_COVER,0.1002,0.2260,U81202010,원변수 단일 채택(파생 폐기)


[ckpt 저장] step15_2_파생검증: 5행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step15_2_파생검증.parquet (0.0 MB)


## STEP 15-3. 추가 후보 발굴 — 잔여 128컬럼 IV 스캔에 노출/대안 태깅
STEP 12-1의 `iv_scan_remain128_entrant`를 로드해 계열별로 태깅한다. **노출 계열(카드·대출·연체·한도)은 대안변수 후보에서 제외** — Track B의 IV 인플레이션 교훈. 비노출 상위가 추가 후보다.

In [17]:
iv_scan = load_ckpt('iv_scan_remain128_entrant')

def classify(col):
    c = col.replace('_was_missing','')
    if c.startswith(('C1M','C18','C1L','C1Z')): return '노출(카드)'
    if c.startswith(('L10','L90','L1Z')):        return '노출(대출)'
    if c.startswith('D10') or c.startswith('D1013'): return '노출(연체이력)'
    if c.startswith('U8'):  return '대안(자산·거주)'
    if c.startswith('AL0'): return '대안(차량·이력)'
    if c.startswith('AS1'): return '대안(거주평형)'
    if c.startswith('AP0'): return '대안(공적보험)'
    if c.startswith('PERF'): return '타겟(사용금지)'
    return '기타'

iv_scan['계열'] = iv_scan['col'].map(classify)
iv_scan['후보여부'] = iv_scan['계열'].str.startswith('대안')
print('[계열별 IV 상위 요약]')
display(iv_scan.groupby('계열')['IV_entrant'].agg(n='size', max='max', mean='mean').round(4))
print('\n[대안 계열 IV 상위 15 — 추가 채택 후보]')
alt_top = iv_scan[iv_scan['후보여부']].sort_values('IV_entrant', ascending=False).head(15)
display(alt_top)
print('\n※ 노출 계열 상위는 모델 C(CB 상한선) 전용 — 모델 A/B 피처 금지')

[ckpt 로드] iv_scan_remain128_entrant: 138행, 0.00 GB
[계열별 IV 상위 요약]


,n,max,mean
계열,,,
기타,2,3.2957,1.6479
노출(대출),44,0.8301,0.2382
노출(연체이력),23,3.8822,1.3613
노출(카드),49,1.3450,0.2513
대안(거주평형),1,0.2746,0.2746
대안(공적보험),2,0.0000,0.0000
대안(자산·거주),10,0.3888,0.2220
대안(차량·이력),5,0.0137,0.0027
타겟(사용금지),2,5.6567,5.3079



[대안 계열 IV 상위 15 — 추가 채택 후보]


,col,IV_entrant,계열,후보여부
40,U81301010,0.388821,대안(자산·거주),True
44,U81305010,0.284215,대안(자산·거주),True
47,AS120G001,0.274557,대안(거주평형),True
50,U81306010,0.247570,대안(자산·거주),True
52,U81302010,0.245633,대안(자산·거주),True
54,U81201010,0.230329,대안(자산·거주),True
57,U81202010,0.226030,대안(자산·거주),True
58,U81204010,0.223842,대안(자산·거주),True
64,U81102010,0.180056,대안(자산·거주),True
67,U81203010,0.168169,대안(자산·거주),True



※ 노출 계열 상위는 모델 C(CB 상한선) 전용 — 모델 A/B 피처 금지


## STEP 15-4. 최종 후보 확정: 다중공선성 + 순환논리 검증 → 목록 저장
후보 풀 = AL012G 3종(기확정) + 15-1 통과 + 15-2 채택 + 15-3 상위. 두 관문을 통과해야 최종 목록에 남는다.
1. **다중공선성**: |r| ≥ 0.8 쌍은 IV 높은 쪽만 대표로 (Track B 7단계와 동일)
2. **순환논리**: TF 정의 5필드 합산치와의 상관 |r| ≥ 0.4면 제외 (Track B 8단계 실측 방식)

In [18]:
# 후보 풀 구성 (15-1·15-3 판정 결과에서 채택/조건부만 + 기확정 AL012G)
pool = list(AL3)
pool += res_151.loc[res_151['판정'].astype(str).str.contains('채택'), 'col'].tolist()
pool += alt_top['col'].str.replace('_was_missing','', regex=False).head(10).tolist()
pool = [c for c in dict.fromkeys(pool) if c in wide.columns]
print(f'후보 풀: {len(pool)}개 → {pool}')

# 수치화 (문자형 A/B/C는 코드 변환)
Xp = pd.DataFrame(index=wide.index)
for c in pool:
    s = wide[c]
    if not pd.api.types.is_numeric_dtype(s):
        s = s.astype('category').cat.codes.replace(-1, np.nan)
    Xp[c] = pd.to_numeric(s, errors='coerce')

# 1) 다중공선성
ivs = {c: iv_num(Xp[c].fillna(0), y) for c in pool}
corr = Xp.corr().abs()
drop = set()
for i, a in enumerate(pool):
    for b in pool[i+1:]:
        if a in drop or b in drop: continue
        if corr.loc[a, b] >= 0.8:
            loser = a if ivs[a] < ivs[b] else b
            drop.add(loser)
            print(f'다중공선성 |r|={corr.loc[a,b]:.2f}: {a} vs {b} → {loser} 제거')

# 2) 순환논리: TF 정의 합산치와 상관
tf_sum = wide[TF_BASE5].fillna(0).sum(axis=1)
circ = {}
for c in [c for c in pool if c not in drop]:
    r = Xp[c].corr(tf_sum)
    circ[c] = r
    if abs(r) >= 0.4:
        drop.add(c); print(f'순환논리 |r|={r:.2f}: {c} 제거')

final = pd.DataFrame({
    'col': [c for c in pool if c not in drop],
}).assign(IV_entrant=lambda t: t['col'].map(ivs).round(4),
          TF정의상관=lambda t: t['col'].map(circ).round(3)).sort_values('IV_entrant', ascending=False)
print(f'\n=== 폴더9 최종 대안변수 후보 {len(final)}개 ===')
display(final)
save_ckpt(final, 'step15_4_최종대안변수목록')
print('\n다음 단계: 이 목록으로 STEP 13 모델 A 피처를 교체해 재실행 → 보고서 성능 섹션 확정')

후보 풀: 16개 → ['AL012G005', 'AL012G011', 'AL012G019', 'U81205010', 'U81303010', 'U81304010', 'AS120G001', 'U81301010', 'U81305010', 'U81306010', 'U81302010', 'U81201010', 'U81202010', 'U81204010', 'U81102010', 'U81203010']
다중공선성 |r|=0.93: U81306010 vs U81204010 → U81204010 제거
다중공선성 |r|=0.94: U81306010 vs U81203010 → U81203010 제거

=== 폴더9 최종 대안변수 후보 14개 ===


,col,IV_entrant,TF정의상관
2,AL012G019,0.4990,0.003
7,U81301010,0.3888,0.044
8,U81305010,0.2842,-0.051
6,AS120G001,0.2746,-0.034
9,U81306010,0.2476,0.026
10,U81302010,0.2456,-0.005
11,U81201010,0.2303,0.016
12,U81202010,0.2260,0.023
13,U81102010,0.1801,0.027
0,AL012G005,0.1687,0.078


[ckpt 저장] step15_4_최종대안변수목록: 14행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step15_4_최종대안변수목록.parquet (0.0 MB)

다음 단계: 이 목록으로 STEP 13 모델 A 피처를 교체해 재실행 → 보고서 성능 섹션 확정


# STEP 13R. 모델 A 재실행 — STEP 15 최종 대안변수 목록으로 피처 교체

기존 STEP 13과의 차이:
1. 피처를 셀에 하드코딩하지 않고 **`step15_4_최종대안변수목록` parquet에서 로드** (15가 갱신되면 이 셀만 다시 누르면 됨)
2. 15-2에서 유일하게 원변수를 이긴 파생 **NONBANK_RATIO를 A-1에 추가** (노출 기반이므로 A-1 '신청정보 포함' 버전 전용 — A-2에는 금지)
3. 문자형 변수(U8 신뢰수준 A/B/C)는 원핫, 수치형(자산 구간·금액)은 그대로
4. 결과를 `step13R_결과` parquet으로 저장 — 리포트의 빈칸을 채우는 공식 숫자

**전제**: STEP 10 실행 + `ent_all` 로드 상태. `wide`가 없어도 배치에서 자동 재구성하므로 15-0을 다시 돌릴 필요 없음.

In [19]:
# ============ STEP 13R ============
import glob
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score

# --- 0) 최종 변수 목록 로드 ---
final = load_ckpt('step15_4_최종대안변수목록')
ALT_FINAL = final['col'].tolist()
print(f'STEP 15 최종 대안변수 {len(ALT_FINAL)}개:', ALT_FINAL)

# --- 1) 진입자 전 연도 와이드 프레임 (배치 병합) ---
wide_all = ent_all.copy()
wide_all['ID'] = wide_all['ID'].astype(str)
for path in sorted(glob.glob(os.path.join(CKPT_DIR, 'batch_*.parquet'))):
    b = pd.read_parquet(path); b['ID'] = b['ID'].astype(str)
    add = ['ID','YEAR'] + [c for c in b.columns if c not in wide_all.columns]
    wide_all = wide_all.merge(b[add], on=['ID','YEAR'], how='left')
print(f'전 연도 프레임: {wide_all.shape[0]:,}행 × {wide_all.shape[1]}열')

# --- 2) 파생 NONBANK_RATIO (15-2 채택) — 전 연도에서 생성 ---
nb_cols = ['L10210800','L10210B00','L90210300','L90210200','L10210M00']
num = wide_all[nb_cols].fillna(0).clip(lower=0).sum(axis=1)
den = wide_all['L10210000'].fillna(0)
wide_all['NONBANK_RATIO'] = np.where(den > 0, (num/den).clip(0,1), -1)   # -1 = 대출없음 범주

# --- 3) t 시점(진입) / t-1 시점(씬파일러 시절) 프레임 ---
ent_t = wide_all[wide_all['IS_ENTRANT']].copy()
prev = wide_all.copy(); prev['YEAR'] = prev['YEAR'] + 1
PREV_COLS = ALT_FINAL   # A-2는 t-1의 대안변수만
ent_t = ent_t.merge(prev[['ID','YEAR'] + PREV_COLS].rename(columns={c: f'{c}_prev' for c in PREV_COLS}),
                    on=['ID','YEAR'], how='left')
print(f"모델 A 모집단: {len(ent_t):,}행, 양성 {int(ent_t.PERF2.sum())} | t-1 병합률 {ent_t[f'{ALT_FINAL[0]}_prev'].notna().mean():.2f}")

def encode(frame, cols):
    X = pd.DataFrame(index=frame.index)
    for c in cols:
        s = frame[c]
        if pd.api.types.is_numeric_dtype(s):
            X[c] = pd.to_numeric(s, errors='coerce').fillna(0)
        else:
            X = pd.concat([X, pd.get_dummies(s.astype(str), prefix=c)], axis=1)
    return X

y = ent_t['PERF2'].astype(int).values
groups = ent_t['ID'].values

def run_cv(X, y, groups, label):
    aps, aucs, lifts = [], [], []
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    for tr_i, va_i in sgkf.split(X, y, groups):
        spw = (y[tr_i]==0).sum() / max((y[tr_i]==1).sum(), 1)
        m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15,
                               min_child_samples=50, scale_pos_weight=spw,
                               random_state=42, n_jobs=-1, verbose=-1)
        m.fit(X.iloc[tr_i], y[tr_i])
        p = m.predict_proba(X.iloc[va_i])[:,1]
        if y[va_i].sum() == 0: continue
        aps.append(average_precision_score(y[va_i], p))
        aucs.append(roc_auc_score(y[va_i], p))
        k = max(int(len(va_i)*0.05), 1)
        lifts.append(y[va_i][np.argsort(-p)[:k]].mean() / max(y[va_i].mean(), 1e-9))
    print(f'[{label}] CV AUPRC {np.mean(aps):.4f}±{np.std(aps):.4f} (기준선 {y.mean():.4f}) | '
          f'AUROC {np.mean(aucs):.4f}±{np.std(aucs):.4f} | top-5% 리프트 {np.mean(lifts):.1f}배')
    return {'label': label, 'auprc': np.mean(aps), 'auprc_sd': np.std(aps),
            'auroc': np.mean(aucs), 'auroc_sd': np.std(aucs),
            'lift5': np.mean(lifts), 'baseline': y.mean(), 'n_feat': X.shape[1]}, m

# --- A-1: 진입 시점 — 최종 대안변수(t) + NONBANK_RATIO + 인구 ---
X1 = encode(ent_t, ALT_FINAL)
X1['NONBANK_RATIO'] = ent_t['NONBANK_RATIO']
X1 = pd.concat([X1, pd.get_dummies(ent_t['GENDER'], prefix='GENDER'),
                    pd.get_dummies(ent_t['AGE_BAND'], prefix='AGE')], axis=1)
r1, m1 = run_cv(X1, y, groups, 'A-1 진입시점(대안변수+신청정보)')

# --- A-2: 진입 전(t-1) — 대안변수만 (순수 대안평가) ---
X2 = encode(ent_t, [f'{c}_prev' for c in ALT_FINAL])
X2 = pd.concat([X2, pd.get_dummies(ent_t['GENDER'], prefix='GENDER'),
                    pd.get_dummies(ent_t['AGE_BAND'], prefix='AGE')], axis=1)
r2, m2 = run_cv(X2, y, groups, 'A-2 진입전 t-1(순수 대안)')

# --- 피처 기여 확인 (A-2 기준: 어떤 대안변수가 일하는가) ---
imp = pd.Series(m2.feature_importances_, index=X2.columns).sort_values(ascending=False)
print('\n[A-2 피처 중요도 상위 12]'); display(imp.head(12).to_frame('importance'))

res = pd.DataFrame([r1, r2])
save_ckpt(res, 'step13R_결과')
print('\n→ 이 두 줄이 리포트 빈칸(스모크 테스트 최종)에 들어갈 공식 숫자')

[ckpt 로드] step15_4_최종대안변수목록: 14행, 0.00 GB
STEP 15 최종 대안변수 14개: ['AL012G019', 'U81301010', 'U81305010', 'AS120G001', 'U81306010', 'U81302010', 'U81201010', 'U81202010', 'U81102010', 'AL012G005', 'U81304010', 'U81303010', 'U81205010', 'AL012G011']
전 연도 프레임: 273,468행 × 175열
모델 A 모집단: 68,677행, 양성 96 | t-1 병합률 1.00
[A-1 진입시점(대안변수+신청정보)] CV AUPRC 0.0029±0.0013 (기준선 0.0014) | AUROC 0.6371±0.0918 | top-5% 리프트 3.1배
[A-2 진입전 t-1(순수 대안)] CV AUPRC 0.0021±0.0009 (기준선 0.0014) | AUROC 0.5679±0.0845 | top-5% 리프트 2.0배

[A-2 피처 중요도 상위 12]


,importance
U81301010_prev,411
U81201010_prev,339
U81202010_prev,320
U81302010_prev,250
U81102010_prev,126
AL012G005_prev,77
AS120G001_prev,72
U81305010_prev,64
AL012G019_prev,43
U81205010_prev_A,43


[ckpt 저장] step13R_결과: 2행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step13R_결과.parquet (0.0 MB)

→ 이 두 줄이 리포트 빈칸(스모크 테스트 최종)에 들어갈 공식 숫자
